In [1]:
conda create -n bot python=3.10 -y


Note: you may need to restart the kernel to use updated packages.


'C:\Users\Gaurav' is not recognized as an internal or external command,
operable program or batch file.


In [1]:
import pygame
def play_music(name):
    pygame.mixer.init()
    pygame.mixer.music.load(name)
    pygame.mixer.music.play()


pygame 2.6.1 (SDL 2.28.4, Python 3.10.20)
Hello from the pygame community. https://www.pygame.org/contribute.html


/opt/anaconda3/envs/myenv/lib/python3.10/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [49]:
play_music("One of the Girls (instrumental).mp3")

In [3]:
!pip install torch
!pip install pandas
!pip install numpy 
!pip install pygame


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 3.2 MB/s  0:00:03 eta 0:00:01


In [3]:
import torch 
import pandas as pd
import numpy as np

In [4]:
from datasets import load_dataset

dataset = load_dataset("imdb")

print(dataset["train"][0])

c:\Users\Gaurav B V\anaconda3\envs\bot\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [5]:
dataset["train"].features

{'text': Value(dtype='string', id=None),
 'label': ClassLabel(names=['neg', 'pos'], id=None)}

Model time

In [6]:
!pip install transformers

In [7]:
!pip install tqdm


In [2]:
import torch
import torch.nn as nn
from transformers import BertConfig
from tqdm.auto import tqdm

class CustomFeedForwardLayer(nn.Module):
    def __init__(self,config, rank = 64):
        super().__init__()

        self.linear1 = nn.Linear(config.hidden_size, config.intermediate_size)
        self.activation = nn.GELU()
        self.linear2 = nn.Linear(config.intermediate_size, config.hidden_size)

        self.A = nn.Parameter(
            torch.randn(config.intermediate_size, config.intermediate_size, rank)
        )
        self.A2 = nn.Parameter(
            torch.randn(config.hidden_size, config.hidden_size, rank)
        )

    def forward(self, x):

        x = self.linear1(x)
        Ax = torch.einsum("bsi,oik->bsok", x, self.A)
        quad = torch.sum(Ax * Ax, dim=-1)
        x = self.activation(x+quad)

        x = self.linear2(x)
        Ax = torch.einsum("bsi,oik->bsok", x, self.A2)
        quad = torch.sum(Ax * Ax, dim=-1)
        return x + quad
    
class BertLayer(nn.Module):
    def __init__(self,config):
        
        super().__init__()

        self.attention = nn.MultiheadAttention(
            embed_dim=config.hidden_size,
            num_heads=config.num_attention_heads,
            batch_first=True)
        
        self.cff = CustomFeedForwardLayer(config)

        self.norm1 = nn.LayerNorm(config.hidden_size)
        self.norm2 = nn.LayerNorm(config.hidden_size)

    def forward(self, x,attention_mask=None):

        attention_output,_ = self.attention(x,x,x)
        x = self.norm1(x + attention_output)

        cffn_output = self.cff(x)
        x = self.norm2(x + cffn_output)
 
        return x
        
class BertEmbeddings(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.word_embeddings = nn.Embedding(
            config.vocab_size, config.hidden_size
        )

        self.position_embeddings = nn.Embedding(
            config.max_position_embeddings, config.hidden_size
        )

        self.layer_norm = nn.LayerNorm(config.hidden_size)

    def forward(self, input_ids):

        seq_length = input_ids.size(1)

        position_ids = torch.arange(
            seq_length, device=input_ids.device
        ).unsqueeze(0)

        word_embeddings = self.word_embeddings(input_ids)
        position_embeddings = self.position_embeddings(position_ids)

        embeddings = word_embeddings + position_embeddings
        embeddings = self.layer_norm(embeddings)

        return embeddings
    

class BertEncoder(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.layers = nn.ModuleList([
            BertLayer(config) for _ in range(config.num_hidden_layers)
        ])

    def forward(self, x, attention_mask=None):
        for layer in self.layers:
            x = layer(x, attention_mask)
        return x


class MyBertModel(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.embeddings = BertEmbeddings(config)
        self.encoder = BertEncoder(config)
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size)

    def forward(self, input_ids, attention_mask=None,embed = False):

        x = self.embeddings(input_ids)
        
        if embed :
            return x 
        
        x = self.encoder(x, attention_mask)
        logits = self.lm_head(x)


        return logits

        # return x


/opt/anaconda3/envs/myenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
config = BertConfig(
    vocab_size=30522,
    hidden_size=256,
    num_hidden_layers=12,
    num_attention_heads=4,
    intermediate_size=256,
    output_dim = 256
)

model = MyBertModel(config)

In [4]:
import torch
import torch.nn as nn
from torch.optim import AdamW

# 1. Device configuration (CUDA, MPS for Mac, or CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
model.to(device)

# 2. Loss Function (CrossEntropy for Masked Language Modeling or Classification)
criterion = nn.CrossEntropyLoss().to(device)

# 3. Optimizer (Weight decay is crucial for Transformers)
optimizer = AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)

In [11]:
from datasets import load_dataset

dataset = load_dataset("imdb")

print(dataset["train"][0])

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [ ]:
from transformers import BertTokenizer
from torch.utils.data import DataLoader

# Load the standard BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_function(examples):
    # Padding and truncation are essential for batching
    return tokenizer(
        examples["text"], 
        padding="max_length", 
        truncation=True, 
        max_length=128 # You can increase this to 256 to match your config
    )

# Apply tokenization to all splits (train, test)
tokenized_datasets = dataset.map(tokenize_function, batched=True)

In [13]:
dataset["train"][3]

{'text': "This film was probably inspired by Godard's Masculin, féminin and I urge you to see that film instead.<br /><br />The film has two strong elements and those are, (1) the realistic acting (2) the impressive, undeservedly good, photo. Apart from that, what strikes me most is the endless stream of silliness. Lena Nyman has to be most annoying actress in the world. She acts so stupid and with all the nudity in this film,...it's unattractive. Comparing to Godard's film, intellectuality has been replaced with stupidity. Without going too far on this subject, I would say that follows from the difference in ideals between the French and the Swedish society.<br /><br />A movie of its time, and place. 2/10.",
 'label': 0}

In [14]:
tokenized_datasets["train"]

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 25000
})

In [34]:
raw_input_ids = tokenized_datasets["test"][0]["input_ids"]

# 2. Convert to Tensor, Add Batch Dim (unsqueeze), and move to Device
input_tensor = torch.tensor(raw_input_ids).unsqueeze(0).to(device)

# 3. Now run the model
output = model(input_tensor)
print(output)

tensor([[[  4.9132, -12.2484, -11.4705,  ..., -11.2067, -12.6627, -11.3982],
         [  4.9132, -12.2484, -11.4705,  ..., -11.2067, -12.6628, -11.3982],
         [  4.9132, -12.2484, -11.4705,  ..., -11.2067, -12.6627, -11.3982],
         ...,
         [  4.9132, -12.2484, -11.4705,  ..., -11.2067, -12.6627, -11.3982],
         [  4.9132, -12.2484, -11.4705,  ..., -11.2067, -12.6627, -11.3982],
         [  4.9132, -12.2484, -11.4705,  ..., -11.2067, -12.6627, -11.3982]]],
       device='mps:0', grad_fn=<LinearBackward0>)


Training loop

In [1]:
from datasets import load_dataset
import re

dataset = load_dataset("wikitext", "wikitext-103-raw-v1")

def clean(example):
    text = example["text"]

    # remove wiki headings
    text = re.sub(r"=+ .*? =+", "", text)
    text = re.sub(r"@.@" , "", text)

    # whitespace cleanup
    text = re.sub(r"\s+", " ", text).strip()

    return {"text": text}
dataset = dataset.map(clean)

cleaned_dataset = dataset["train"].filter(
    lambda x: len(x["text"]) > 50
)

c:\Users\Gaurav B V\anaconda3\envs\bot\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from transformers import BertTokenizer
from torch.utils.data import DataLoader

# Load the standard BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_function(examples):
    # Padding and truncation are essential for batching
    return tokenizer(
        examples["text"], 
        padding="max_length", 
        truncation=True, 
        max_length=128 # You can increase this to 256 to match your config
    )

# Apply tokenization to all splits (train, test)
# tokenized_datasets = dataset.map(tokenize_function, batched=True)
tokenized_datasets = cleaned_dataset.map(tokenize_function,batched=True)

Map: 100%|██████████| 788438/788438 [01:49<00:00, 7179.05 examples/s]


In [4]:
tokenized_datasets

Dataset({
    features: ['text', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 788438
})

In [ ]:
total_steps = 100000

In [ ]:
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=1000,
    num_training_steps=total_steps
)

In [ ]:
vocab_size = 30522
mask_token_id = 105  

test_test = tokenized_datasets[0:10000]["input_ids"]

epochs = 2

total_loss = 0
model.train()
for epoch in range(epochs):
    for i in test_test:
        for j in range(2):

            optimizer.zero_grad()

            data = torch.tensor(i).unsqueeze(0).to(device)
            
            labels = data.clone()
            labels = labels.to(device)
            probability_matrix = torch.full(labels.shape, 0.15)
            masked_indices = torch.bernoulli(probability_matrix).bool()
            
            inputs = data.clone()

            inputs[masked_indices] = mask_token_id 
            inputs_tensor = torch.tensor(inputs).to(device)
            
            outputs = model(inputs_tensor)


            loss = criterion(outputs[masked_indices], labels[masked_indices])
            
            loss.backward()
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()          
    print(f"Epoch {epoch},total_loss Loss: {total_loss:.4f}  , average loss is {total_loss/len(test_test)}")
play_music("One of the Girls (instrumental).mp3")

NameError: name 'model' is not defined

In [12]:
outputs[masked_indices].shape,labels[masked_indices].shape

(torch.Size([18, 30522]), torch.Size([18]))

In [13]:
from datasets import load_dataset

ds = load_dataset("sentence-transformers/stsb")

In [14]:
from transformers import BertTokenizer
from torch.utils.data import DataLoader

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_function_text(text):
    return tokenizer(
        text,
        padding="max_length", 
        truncation=True, 
        max_length=128 
    )

In [15]:
ds["train"],ds["train"][0]

(Dataset({
     features: ['sentence1', 'sentence2', 'score'],
     num_rows: 5749
 }),
 {'sentence1': 'A plane is taking off.',
  'sentence2': 'An air plane is taking off.',
  'score': 1.0})

In [25]:
validation_inputs = []
for i in ds["train"]:
    i["sentence1_tokens"] = tokenize_function_text(i["sentence1"])
    i["sentence2_tokens"] = tokenize_function_text(i["sentence2"])
    validation_inputs.append(i)


In [19]:
import torch
import torch.nn.functional as F
def sentance_embeddings(text):    
    raw_input_ids = text
    input_tensor1 = torch.tensor(raw_input_ids).unsqueeze(0).to(device)
    output1 = model(input_tensor1,None,True)
    return  output1

def cosine_similarity(embeddings,embeddings2):
# embedding shape: [1, 128, 256]
    embeddings = embeddings.mean(dim=1)   # [1,256]
    embeddings2 = embeddings2.mean(dim=1)
    print(embeddings.shape)
    sim = F.cosine_similarity(embeddings, embeddings2)
    return sim


In [28]:
import torch
import torch.nn as nn
from torch.optim import AdamW
import math

# 1. Device configuration (CUDA, MPS for Mac, or CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
loss = 0
for i in validation_inputs:
    raw_input_ids = i['sentence1_tokens']['input_ids']
    output1 = sentance_embeddings(raw_input_ids)
    output2 = sentance_embeddings( i['sentence2_tokens']['input_ids'])
    # input_tensor1 = torch.tensor(raw_input_ids).unsqueeze(0).to(device)
    # print(input_tensor1)
    # output1 = model(input_tensor1,None,True)
    sim = cosine_similarity(output1,output2)
    print(sim,i['score'])
    
    loss+= abs(sim-i['score'])
    print(output2.shape)
    print(sim.item())


torch.Size([1, 256])
tensor([0.9999], device='mps:0', grad_fn=<SumBackward1>) 1.0
torch.Size([1, 128, 256])
0.9998824000358582
torch.Size([1, 256])
tensor([1.0000], device='mps:0', grad_fn=<SumBackward1>) 0.76
torch.Size([1, 128, 256])
0.9999575018882751
torch.Size([1, 256])
tensor([0.9996], device='mps:0', grad_fn=<SumBackward1>) 0.76
torch.Size([1, 128, 256])
0.9996113777160645
torch.Size([1, 256])
tensor([0.9999], device='mps:0', grad_fn=<SumBackward1>) 0.52
torch.Size([1, 128, 256])
0.9999249577522278
torch.Size([1, 256])
tensor([1.0000], device='mps:0', grad_fn=<SumBackward1>) 0.85
torch.Size([1, 128, 256])
0.9999622702598572
torch.Size([1, 256])
tensor([0.9999], device='mps:0', grad_fn=<SumBackward1>) 0.85
torch.Size([1, 128, 256])
0.9999312162399292
torch.Size([1, 256])
tensor([0.9999], device='mps:0', grad_fn=<SumBackward1>) 0.1
torch.Size([1, 128, 256])
0.9999265074729919
torch.Size([1, 256])
tensor([0.9999], device='mps:0', grad_fn=<SumBackward1>) 0.32
torch.Size([1, 128, 256

In [30]:
loss/len(validation_inputs)

tensor([0.4593], device='mps:0', grad_fn=<DivBackward0>)

In [26]:
validation_inputs

[{'sentence1': 'A plane is taking off.',
  'sentence2': 'An air plane is taking off.',
  'score': 1.0,
  'sentence1_tokens': {'input_ids': [101, 1037, 4946, 2003, 2635, 2125, 1012, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 

In [27]:
import torch
import torch.nn as nn
from torch.optim import AdamW
import math

# 1. Device configuration (CUDA, MPS for Mac, or CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
loss = 0
for i in validation_inputs:
    raw_input_ids = i['sentence1_tokens']['input_ids']
    output1 = sentance_embeddings(raw_input_ids)
    output2 = sentance_embeddings( i['sentence2_tokens']['input_ids'])
    print(raw_input_ids)
    print(i['sentence2_tokens']['input_ids'])
    print(output1)
    print(output2)
    break
    # input_tensor1 = torch.tensor(raw_input_ids).unsqueeze(0).to(device)
    # print(input_tensor1)
    # output1 = model(input_tensor1,None,True)
    sim = cosine_similarity(output1,output2)
    print(sim,i['score'])
    
    loss+= abs(sim-i['score'])
    print(output2.shape)
    print(sim.item())


[101, 1037, 4946, 2003, 2635, 2125, 1012, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[101, 2019, 2250, 4946, 2003, 2635, 2125, 1012, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
tensor([[[ 1.0585, -2.4314, -0.4681,  ...,  0.6211,  0.7857, -1.1160],
         [ 0.2345, -0.6300, -1.0970,  ...,  1.3045, -0.8028,  1.5741],
         [-0.9126,  0.5485, -1.2544,  ...

In [30]:
loss/len(validation_inputs)

tensor([0.4598], device='mps:0', grad_fn=<DivBackward0>)

In [138]:
raw_input_ids = tokenized_datasets["train"][3]["input_ids"]

# 2. Convert to Tensor, Add Batch Dim (unsqueeze), and move to Device
input_tensor1 = torch.tensor(raw_input_ids).unsqueeze(0).to(device)

# 3. Now run the model
output1 = model(input_tensor1,None,True)
print(output1.shape)

torch.Size([1, 128, 256])


In [139]:
output1

tensor([[[-0.9193,  0.0243,  1.1160,  ..., -0.0990, -0.6564,  0.0703],
         [-1.2124, -1.7314,  0.7804,  ..., -0.6422,  0.2330,  0.2984],
         [-0.7370,  0.2628,  1.6661,  ...,  1.9095, -0.2090, -2.3254],
         ...,
         [-0.2209, -0.6102, -1.6574,  ...,  0.8126,  1.5709,  0.2948],
         [ 0.4811, -1.7179, -0.0443,  ...,  0.3519,  1.3095,  0.8031],
         [ 0.3759,  0.5002, -0.3975,  ...,  0.2805, -0.5343, -0.0872]]],
       device='cuda:0', grad_fn=<NativeLayerNormBackward0>)

In [114]:
tokenized_datasets["test"][0]

{'text': '',
 'input_ids': [101,
  102,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0],
 'token_type_ids': [0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
 

In [141]:
raw_input_ids = tokenized_datasets["test"][1]["input_ids"]

# 2. Convert to Tensor, Add Batch Dim (unsqueeze), and move to Device
input_tensor = torch.tensor(raw_input_ids).unsqueeze(0).to(device)

# 3. Now run the model
output = model(input_tensor,None,True)
print(output.shape)

torch.Size([1, 128, 256])


In [142]:
output

tensor([[[-0.9193,  0.0243,  1.1160,  ..., -0.0990, -0.6564,  0.0703],
         [-0.6171, -0.0406,  1.8661,  ..., -0.9518,  1.1199,  0.6129],
         [-0.6251,  0.5475,  0.6828,  ...,  1.7461,  0.4007, -2.5632],
         ...,
         [ 0.4509, -0.3225,  0.7098,  ...,  0.3701,  2.7233,  1.1461],
         [ 0.7654, -1.3788,  1.1469,  ..., -0.3917,  2.3414,  0.8888],
         [ 1.8654,  1.0061, -0.3746,  ...,  0.9155,  2.2126, -0.1477]]],
       device='cuda:0', grad_fn=<NativeLayerNormBackward0>)

In [143]:
import torch.nn as nn

cos = nn.CosineSimilarity(dim=2) # Dim 2 is the '256' feature dimension

output = cos(output1, output )
print(output.shape) # Result: torch.Size([1, 128])

torch.Size([1, 128])


In [145]:
output

tensor([[1.0000, 0.5274, 0.5701, 0.4283, 0.4543, 0.4903, 0.4467, 0.4823, 0.5990,
         0.5734, 0.5109, 0.4719, 0.5195, 0.5010, 0.5112, 0.5586, 0.5840, 0.5135,
         0.5311, 0.5486, 0.4991, 0.5210, 0.4897, 0.5041, 0.5298, 0.4913, 0.4624,
         0.5484, 0.4923, 0.4745, 0.4874, 0.4695, 0.5349, 0.4899, 0.4490, 0.4395,
         0.5042, 0.4770, 0.5251, 0.5055, 0.5521, 0.4567, 0.4463, 0.4716, 0.5359,
         0.5560, 0.4930, 0.4814, 0.5153, 0.5104, 0.4911, 0.5174, 0.5104, 0.5560,
         0.5210, 0.4539, 0.5052, 0.4756, 0.5077, 0.5635, 0.5235, 0.5820, 0.4955,
         0.5763, 0.5859, 0.5044, 0.5554, 0.4946, 0.4122, 0.4603, 0.5232, 0.5357,
         0.4614, 0.4657, 0.4929, 0.4720, 0.5069, 0.4977, 0.4605, 0.4599, 0.5284,
         0.5500, 0.4239, 0.5156, 0.5358, 0.5631, 0.5697, 0.4782, 0.4978, 0.5832,
         0.4923, 0.5126, 0.5473, 0.4770, 0.4837, 0.5605, 0.5444, 0.4793, 0.4903,
         0.4935, 0.3896, 0.4971, 0.5707, 0.5184, 0.5515, 0.5571, 0.5243, 0.5380,
         0.4982, 0.5475, 0.5

In [101]:
import torch
import torch.nn.functional as F

In [102]:
x = torch.randn(1, 128, 256)

# Step 1: Remove the batch dimension for easier math -> [128, 256]
vectors = x.squeeze(0) 

# Step 2: Normalize the vectors (Magnitude = 1)
# This is crucial because Cosine Similarity is just a Dot Product of normalized vectors
norm_vectors = F.normalize(vectors, p=2, dim=1)

# Step 3: Matrix Multiply by its own transpose
# [128, 256] @ [256, 128] = [128, 128]
sim_matrix = torch.mm(norm_vectors, norm_vectors.t())

print(sim_matrix.shape) # Result: torch.Size([128, 128])

torch.Size([128, 128])


In [103]:
import torch.nn as nn

cos = nn.CosineSimilarity(dim=2) # Dim 2 is the '256' feature dimension

x1 = torch.randn(1, 128, 256)
x2 = torch.randn(1, 128, 256)

output = cos(x1, x2)
print(output.shape) # Result: torch.Size([1, 128])

torch.Size([1, 128])


In [111]:
output


tensor([[1.0000, 0.5519, 0.4880, 0.4514, 0.4867, 0.5031, 0.5146, 0.5004, 1.0000,
         1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
         1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
         1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
         1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
         1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
         1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
         1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
         1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
         1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
         1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
         1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
         1.0000, 1.0000, 1.0

In [82]:
print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

Epoch 99, Loss: 0.6762
